# 06 — Finalna Evaluacija i Poređenje Svih Modela
**Cilj:** Skupiti rezultate svih modela iz prethodnih notebooka, napraviti detaljnu komparativnu analizu i izvući zaključke.

**Modeli koji se porede:**
1. MLP + Class Weight (baseline)
2. MLP + SMOTE (baseline)
3. MLP + SMOTE+Under (baseline)
4. Deep BN MLP + Focal Loss
5. ResNet MLP + Focal Loss
6. Tuned MLP (Bayesian Optimization)
7. Autoencoder (unsupervised)

---

## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import json, os, warnings

import tensorflow as tf
from tensorflow import keras

from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    classification_report, confusion_matrix,
    f1_score, precision_score, recall_score,
    roc_curve, precision_recall_curve
)

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

PROC    = '../data/processed/'
MODELS  = '../models/'
RESULTS = '../results/'

print('✅ Setup OK')

## 1. Učitavanje podataka i modela

In [ ]:
X_val   = np.load(f'{PROC}X_val.npy')
X_test  = np.load(f'{PROC}X_test.npy')
y_val   = np.load(f'{PROC}y_val.npy')
y_test  = np.load(f'{PROC}y_test.npy')

print(f'Test set: {X_test.shape} | Fraud: {y_test.sum()} ({y_test.mean()*100:.2f}%)')

In [ ]:
# Focal Loss mora biti registrovan pre učitavanja modela koji ga koriste
class FocalLoss(keras.losses.Loss):
    def __init__(self, gamma=2.0, alpha=0.25, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.alpha = alpha

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        bce    = -y_true * tf.math.log(y_pred) - (1 - y_true) * tf.math.log(1 - y_pred)
        p_t    = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        alpha_t = y_true * self.alpha + (1 - y_true) * (1 - self.alpha)
        return tf.reduce_mean(alpha_t * tf.pow(1 - p_t, self.gamma) * bce)

    def get_config(self):
        c = super().get_config()
        c.update({'gamma': self.gamma, 'alpha': self.alpha})
        return c


# Učitavanje svih sačuvanih modela
custom_objects = {'FocalLoss': FocalLoss}

model_files = {
    'MLP + Class Weight':    'mlp_classweight.keras',
    'MLP + SMOTE':           'mlp_smote.keras',
    'MLP + SMOTE+Under':     'mlp_combined.keras',
    'Deep BN MLP + Focal':   'mlp_bn.keras',
    'ResNet MLP + Focal':    'mlp_resnet.keras',
    'Tuned MLP (BayesOpt)':  'mlp_tuned.keras',
    'Autoencoder':           'autoencoder.keras',
}

models = {}
for name, fname in model_files.items():
    path = f'{MODELS}{fname}'
    if os.path.exists(path):
        models[name] = keras.models.load_model(path, custom_objects=custom_objects)
        print(f'✅ Učitan: {name}')
    else:
        print(f'⚠️  Nije pronađen: {name} ({path})')

## 2. Evaluacija svih modela

In [ ]:
def get_scores(model, X, is_autoencoder=False):
    """Vraća anomaly/probability scores."""
    if is_autoencoder:
        X_rec = model.predict(X, verbose=0)
        return np.mean(np.power(X - X_rec, 2), axis=1)
    else:
        return model.predict(X, verbose=0).ravel()


def best_threshold_f1(scores, y_true):
    """Optimalni threshold po F1 na val setu."""
    ths = np.linspace(scores.min(), scores.max(), 200)
    f1s = [f1_score(y_true, (scores >= t).astype(int)) for t in ths]
    return ths[np.argmax(f1s)]


all_results = []
all_curves  = {}   # za ROC/PR plot

for name, model in models.items():
    is_ae = 'Autoencoder' in name
    
    val_scores  = get_scores(model, X_val, is_ae)
    test_scores = get_scores(model, X_test, is_ae)
    
    t = best_threshold_f1(val_scores, y_val)
    y_pred = (test_scores >= t).astype(int)
    
    roc_auc = roc_auc_score(y_test, test_scores)
    pr_auc  = average_precision_score(y_test, test_scores)
    f1      = f1_score(y_test, y_pred)
    prec    = precision_score(y_test, y_pred, zero_division=0)
    rec     = recall_score(y_test, y_pred)
    
    fpr, tpr, _ = roc_curve(y_test, test_scores)
    p, r, _     = precision_recall_curve(y_test, test_scores)
    all_curves[name] = {'fpr': fpr, 'tpr': tpr, 'prec': p, 'rec': r}
    
    all_results.append({
        'Model': name,
        'ROC-AUC': round(roc_auc, 4),
        'PR-AUC':  round(pr_auc, 4),
        'F1':      round(f1, 4),
        'Precision': round(prec, 4),
        'Recall':  round(rec, 4),
        'Threshold': round(t, 4),
        'Tip':     'Unsupervised' if is_ae else 'Supervised'
    })
    print(f'{name:<30} ROC={roc_auc:.4f}  PR={pr_auc:.4f}  F1={f1:.4f}  Rec={rec:.4f}')

results_df = pd.DataFrame(all_results).set_index('Model')

## 3. Sumarni pregled metrika

In [ ]:
print('\n=== FINALNE METRIKE — SVI MODELI ===')
display(
    results_df[['ROC-AUC', 'PR-AUC', 'F1', 'Precision', 'Recall', 'Tip']]
    .sort_values('PR-AUC', ascending=False)
    .style
    .highlight_max(color='#90EE90', subset=['ROC-AUC', 'PR-AUC', 'F1', 'Recall'])
    .highlight_min(color='#FFCCCC', subset=['ROC-AUC', 'PR-AUC', 'F1'])
    .bar(subset=['PR-AUC'], color='#AED6F1')
)

results_df.to_csv(f'{RESULTS}06_final_results.csv')
print('\nRezultati sačuvani u ../results/06_final_results.csv')

## 4. ROC i PR Krive — Svi Modeli

In [ ]:
colors = plt.cm.tab10(np.linspace(0, 1, len(all_curves)))

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for (name, curves), color in zip(all_curves.items(), colors):
    roc_auc = results_df.loc[name, 'ROC-AUC']
    pr_auc  = results_df.loc[name, 'PR-AUC']
    
    axes[0].plot(curves['fpr'], curves['tpr'], lw=2, color=color,
                 label=f'{name} ({roc_auc:.3f})')
    axes[1].plot(curves['rec'], curves['prec'], lw=2, color=color,
                 label=f'{name} ({pr_auc:.3f})')

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
axes[0].set_xlabel('False Positive Rate', fontsize=12)
axes[0].set_ylabel('True Positive Rate', fontsize=12)
axes[0].set_title('ROC Krive — Svi Modeli', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=8, loc='lower right')

baseline_pr = y_test.mean()
axes[1].axhline(baseline_pr, color='black', linestyle='--', alpha=0.5,
                label=f'Random ({baseline_pr:.3f})')
axes[1].set_xlabel('Recall', fontsize=12)
axes[1].set_ylabel('Precision', fontsize=12)
axes[1].set_title('Precision-Recall Krive — Svi Modeli', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=8, loc='upper right')

plt.tight_layout()
plt.savefig(f'{RESULTS}06_all_roc_pr_curves.png', bbox_inches='tight')
plt.show()

## 5. Vizualizacija metrika — Radar Chart i Heatmap

In [ ]:
# ---- Heatmap metrika ----
metrics_cols = ['ROC-AUC', 'PR-AUC', 'F1', 'Precision', 'Recall']
heatmap_data = results_df[metrics_cols].sort_values('PR-AUC', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.heatmap(
    heatmap_data, annot=True, fmt='.4f',
    cmap='YlOrRd', vmin=0.5, vmax=1.0,
    linewidths=0.5, ax=axes[0]
)
axes[0].set_title('Heatmap Performansi Modela', fontweight='bold', fontsize=13)
axes[0].tick_params(axis='x', rotation=30)

# ---- Grouped Bar Chart — ključne metrike ----
key_metrics = ['PR-AUC', 'F1', 'Recall']
models_list = heatmap_data.index.tolist()
x = np.arange(len(models_list))
w = 0.25
bar_colors = ['#3498db', '#e74c3c', '#2ecc71']

for i, (metric, color) in enumerate(zip(key_metrics, bar_colors)):
    axes[1].bar(x + i * w, heatmap_data[metric], w, label=metric, color=color, alpha=0.85)

axes[1].set_xticks(x + w)
axes[1].set_xticklabels(models_list, rotation=20, ha='right', fontsize=9)
axes[1].set_ylim(0.5, 1.01)
axes[1].set_ylabel('Score')
axes[1].set_title('PR-AUC, F1, Recall — Poređenje', fontweight='bold', fontsize=13)
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{RESULTS}06_metrics_heatmap.png', bbox_inches='tight')
plt.show()

## 6. Confusion Matrice — Svi Modeli

In [ ]:
n_models = len(models)
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, (name, model) in enumerate(models.items()):
    is_ae = 'Autoencoder' in name
    test_scores = get_scores(model, X_test, is_ae)
    val_scores  = get_scores(model, X_val, is_ae)
    
    t = best_threshold_f1(val_scores, y_val)
    y_pred = (test_scores >= t).astype(int)
    cm = confusion_matrix(y_test, y_pred)
    
    # Normalizovana CM
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud'])
    
    f1  = results_df.loc[name, 'F1']
    rec = results_df.loc[name, 'Recall']
    axes[i].set_title(f'{name}\nF1={f1:.3f}  Recall={rec:.3f}', fontsize=9, fontweight='bold')
    axes[i].set_ylabel('Stvarno')
    axes[i].set_xlabel('Predviđeno')

# Sakrij prazan subplot ako postoji
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.suptitle('Confusion Matrice — Svi Modeli (Test Set)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS}06_confusion_matrices.png', bbox_inches='tight')
plt.show()

## 7. Analiza Cost-Benefit (Poslovni kontekst)

In [ ]:
# Pretpostavke o troškovima (realan poslovni scenario)
# FN (propuštena prevara) = skupo: prosečan izgubljen iznos
# FP (lažni alarm) = umereno skupo: ručna provjera, nezadovoljstvo korisnika
COST_FN = 200   # €  — prosečna propuštena prevara
COST_FP = 10    # €  — operativni trošak lažnog alarma

print(f'=== COST-BENEFIT ANALIZA ===')
print(f'Pretpostavke: FN trošak = {COST_FN}€, FP trošak = {COST_FP}€')
print()

cost_results = []
n_fraud = y_test.sum()
n_legit = (y_test == 0).sum()

for name, model in models.items():
    is_ae = 'Autoencoder' in name
    scores = get_scores(model, X_test, is_ae)
    val_s  = get_scores(model, X_val, is_ae)
    t = best_threshold_f1(val_s, y_val)
    y_pred = (scores >= t).astype(int)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    total_cost = fn * COST_FN + fp * COST_FP
    # Baseline: niko nije flagovan kao fraud (sve propustimo)
    baseline_cost = n_fraud * COST_FN
    savings = baseline_cost - total_cost
    
    cost_results.append({
        'Model': name, 'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
        'FN Cost (€)': fn * COST_FN,
        'FP Cost (€)': fp * COST_FP,
        'Total Cost (€)': total_cost,
        'Savings vs Baseline (€)': savings
    })

cost_df = pd.DataFrame(cost_results).set_index('Model')
display(cost_df.sort_values('Total Cost (€)')
        .style.highlight_min(color='#90EE90', subset=['Total Cost (€)'])
              .highlight_max(color='#90EE90', subset=['Savings vs Baseline (€)']))

# Bar plot ukupnih troškova
fig, ax = plt.subplots(figsize=(12, 5))
sorted_cost = cost_df.sort_values('Total Cost (€)', ascending=True)
bars = ax.barh(sorted_cost.index, sorted_cost['Total Cost (€)'],
               color='salmon', edgecolor='black')
ax.axvline(n_fraud * COST_FN, color='red', linestyle='--', lw=2,
           label=f'Baseline (nema detekcije) = {n_fraud * COST_FN:,}€')
for bar, val in zip(bars, sorted_cost['Total Cost (€)']):
    ax.text(val + 200, bar.get_y() + bar.get_height()/2,
            f'{val:,.0f}€', va='center', fontsize=9)
ax.set_xlabel('Ukupan Trošak (€)')
ax.set_title('Cost-Benefit Analiza — Ukupan Trošak po Modelu', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(f'{RESULTS}06_cost_benefit.png', bbox_inches='tight')
plt.show()

## 8. Zaključci i Preporuke

In [ ]:
best_pr  = results_df['PR-AUC'].idxmax()
best_f1  = results_df['F1'].idxmax()
best_rec = results_df['Recall'].idxmax()

print('=' * 65)
print('FINALNI ZAKLJUČCI')
print('=' * 65)
print()
print('1. KLJUČNE METRIKE za imbalanced fraud detection:')
print('   PR-AUC > F1 > Recall — ROC-AUC je manje informativan')
print('   zbog ekstremne nebalansiranosti (0.17% fraud)')
print()
print(f'2. BEST PR-AUC:   {best_pr} ({results_df.loc[best_pr,"PR-AUC"]:.4f})')
print(f'   BEST F1-Score: {best_f1} ({results_df.loc[best_f1,"F1"]:.4f})')
print(f'   BEST Recall:   {best_rec} ({results_df.loc[best_rec,"Recall"]:.4f})')
print()
print('3. TRETMAN NEBALANSIRANOSTI:')
print('   - class_weight je generalno dobar polazni pristup')
print('   - SMOTE pomaže ali rizikuje overfitting na sintetičkim podacima')
print('   - Focal Loss značajno poboljšava Recall')
print()
print('4. ARHITEKTURA:')
print('   - Batch Normalization ubrzava trening i stabilizuje')
print('   - Residual connections pomažu kod dubljh mreža')
print('   - Bayesian Opt. pronalazi bolje HP od ručnog podešavanja')
print()
print('5. AUTOENCODER:')
print('   - Dobar kao komplementarni pristup')
print('   - Koristan kada nema labeled podataka')
print('   - Slabiji Recall od supervised modela')
print()
print('6. PREPORUKA za produkciju:')
print(f'   → {best_pr}')
print('   → Optimizovati threshold po Recall (ne F1!) za max detekciju')
print('   → Razmotriti ensemble: Supervised + Autoencoder score')
print('=' * 65)

## 9. Finalni Summary Dashboard

In [ ]:
fig = plt.figure(figsize=(20, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# PR-AUC ranking
ax1 = fig.add_subplot(gs[0, 0])
sorted_pr = results_df['PR-AUC'].sort_values(ascending=True)
colors_bar = ['#e74c3c' if v == sorted_pr.max() else '#3498db' for v in sorted_pr]
ax1.barh(sorted_pr.index, sorted_pr, color=colors_bar, edgecolor='black')
ax1.set_title('PR-AUC Ranking', fontweight='bold')
ax1.set_xlim(0.6, 1.0)
ax1.tick_params(labelsize=8)
for i, v in enumerate(sorted_pr):
    ax1.text(v + 0.003, i, f'{v:.4f}', va='center', fontsize=8)

# F1 ranking
ax2 = fig.add_subplot(gs[0, 1])
sorted_f1 = results_df['F1'].sort_values(ascending=True)
colors_f1 = ['#e74c3c' if v == sorted_f1.max() else '#2ecc71' for v in sorted_f1]
ax2.barh(sorted_f1.index, sorted_f1, color=colors_f1, edgecolor='black')
ax2.set_title('F1 Ranking', fontweight='bold')
ax2.set_xlim(0.6, 1.0)
ax2.tick_params(labelsize=8)
for i, v in enumerate(sorted_f1):
    ax2.text(v + 0.003, i, f'{v:.4f}', va='center', fontsize=8)

# Recall ranking
ax3 = fig.add_subplot(gs[0, 2])
sorted_rec = results_df['Recall'].sort_values(ascending=True)
colors_rec = ['#e74c3c' if v == sorted_rec.max() else '#e67e22' for v in sorted_rec]
ax3.barh(sorted_rec.index, sorted_rec, color=colors_rec, edgecolor='black')
ax3.set_title('Recall Ranking', fontweight='bold')
ax3.set_xlim(0.6, 1.0)
ax3.tick_params(labelsize=8)
for i, v in enumerate(sorted_rec):
    ax3.text(v + 0.003, i, f'{v:.4f}', va='center', fontsize=8)

# ROC krive
ax4 = fig.add_subplot(gs[1, 0:2])
colors_lines = plt.cm.tab10(np.linspace(0, 1, len(all_curves)))
for (name, curves), c in zip(all_curves.items(), colors_lines):
    ax4.plot(curves['rec'], curves['prec'], lw=1.5, color=c,
             label=f"{name.split('+')[0].strip()} ({results_df.loc[name,'PR-AUC']:.3f})")
ax4.axhline(y_test.mean(), color='black', linestyle='--', alpha=0.4, label='Random baseline')
ax4.set_xlabel('Recall')
ax4.set_ylabel('Precision')
ax4.set_title('Precision-Recall Krive', fontweight='bold')
ax4.legend(fontsize=7, loc='upper right')

# Tabela metrika
ax5 = fig.add_subplot(gs[1, 2])
ax5.axis('off')
tbl_data = results_df[['PR-AUC', 'F1', 'Recall']].sort_values('PR-AUC', ascending=False)
tbl = ax5.table(
    cellText=tbl_data.values.round(4),
    rowLabels=[n[:20] for n in tbl_data.index],
    colLabels=tbl_data.columns,
    cellLoc='center', loc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)
tbl.scale(1.2, 1.5)
ax5.set_title('Tabela Metrika', fontweight='bold', pad=15)

fig.suptitle('Credit Card Fraud Detection — Finalni Dashboard', fontsize=16, fontweight='bold', y=1.01)
plt.savefig(f'{RESULTS}06_final_dashboard.png', bbox_inches='tight', dpi=150)
plt.show()

print('\n✅ PROJEKAT ZAVRŠEN!')
print(f'Svi rezultati sačuvani u: {RESULTS}')
print(f'Svi modeli sačuvani u:    {MODELS}')